# **📊 Clean & Jerk Dataset — Sync & Summary Explorer**

### 🔧 Colab Setup & Library Init

In [2]:
# @title 🔧 Colab Setup & Library Init { display-mode: "form" }
import importlib, sys, types
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp = types.ModuleType("imp")
        _imp.reload = importlib.reload
        sys.modules["imp"] = _imp

%load_ext autoreload
%autoreload 2

import importlib.util, os, subprocess

# 1. Colab-specific: mount Google Drive & configure renderers
try:
    from google.colab import drive, data_table
    import plotly.io as pio
    plotly_renderer = "colab" # @param ["colab", "vscode", "notebook_connected", "iframe"]
    pio.renderers.default = plotly_renderer
    data_table.enable_dataframe_formatter()
    if not os.path.exists("/content/drive/MyDrive"):
        try:
            drive.mount('/content/drive', force_remount=False)
        except Exception as e:
            print(f"⚠️ Drive not mounted ({e}). Mount via Colab sidebar if needed.")
except ImportError:
    pass

# 2. Ensure dependencies are installed
if importlib.util.find_spec("h5py") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "h5py"])

# 3. Clone (or pull) public repo into /content/Wearable-Biomechanics and add to sys.path
repo_dir = "/content/Wearable-Biomechanics" # @param {type:"string"}
git_branch = "dataset_explorer" # @param {type:"string"}

if os.path.exists(os.path.join(".", "src", "__init__.py")):
    repo_root = os.path.abspath(".")
else:
    if not os.path.exists(repo_dir):
        subprocess.check_call([
            "git", "clone", "-b", git_branch,
            "https://github.com/janithcyapa/Wearable-Biomechanics.git", repo_dir
        ])
    else:
        subprocess.run(["git", "-C", repo_dir, "checkout", git_branch], check=False)
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=False)
        for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
            importlib.reload(sys.modules[_m])
    repo_root = os.path.abspath(repo_dir)

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src import display_table
from src.explorer import DEFAULT_CLEAN_DATASET_DIR, load_summary_tables, load_set_h5
from scripts.sync_summaries_to_h5 import sync_dataset_h5

dataset_dir = DEFAULT_CLEAN_DATASET_DIR # @param {type:"string"}
print(f"📂 Dataset folder: {dataset_dir}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
📂 Dataset folder: /content/drive/MyDrive/Academic/Research/Wearable Sensing/Wearable Biomechanics/Clean&JerkDataset


### 🔄 Sync Summary CSVs → `.h5` & Remove `/fault_log`

Run with `dry_run = True` first to preview counts, then uncheck `dry_run` to apply updates to all `sets/*.h5` files.

In [4]:
# @title 🔄 Sync Summary CSVs to sets/*.h5 & Remove /fault_log { display-mode: "form" }
dry_run = False # @param {type:"boolean"}
remove_fault_log = True # @param {type:"boolean"}
sync_sets = True # @param {type:"boolean"}
sync_participants = True # @param {type:"boolean"}

sync_stats = sync_dataset_h5(
    dataset_dir=dataset_dir,
    remove_fault_log=remove_fault_log,
    sync_sets=sync_sets,
    sync_participants=sync_participants,
    dry_run=dry_run,
)

[UPDATED] Scanned 54 .h5 files | /fault_log removed: 54 | /metadata synced: 54 | /participant_metadata synced: 54


### 📁 Root Summary DataFrames

In [5]:
# @title 📁 Load All Root Summary Tables
tables = load_summary_tables(dataset_dir)
for name, df in tables.items():
    if not df.empty:
        print(f"• {name}: {df.shape[0]} rows × {df.shape[1]} cols")

• participants: 19 rows × 11 cols
• sets: 54 rows × 15 cols
• devices: 26 rows × 10 cols
• phase_markers: 2952 rows × 10 cols
• data_dictionary: 58 rows × 5 cols


In [6]:
# @title 👥 Participants Summary (`participant_summary.csv`)
display_table(tables["participants"], clean_columns=False)

   participant_id                       name  age_years sex  height_m  \
0           P0001               Enuka Mendis         24   M      1.72   
1           P0002      Thevindu Samarasekera         25   M      1.69   
2           P0003             Achalani Royal         23   F      1.60   
3           P0004         Pandula Thennakoon         26   M      1.67   
4           P0005          Induwara Liyanage         21   M      1.70   
5           P0006           Duleetha Malshan         24   M      1.72   
6           P0007         Hivindu Kirulapana         22   M      1.74   
7           P0008          Yewin Dissanayake         22   M      1.70   
8           P0009         Yasith Dissanayake         22   M      1.78   
9           P0010  J W K Mahasen Jayasundara         24   M      1.78   
10          P0011      Nirmal Samarawickrama         22   M      1.73   
11          P0012   J. M. Sudeshe Jayasinghe         23   M      1.56   
12          P0013          Dulan Jayasooriya       

In [7]:
# @title 🏋️ Sets Summary (`sets_summary.csv`)
display_table(tables["sets"], clean_columns=False)

      set_id participant_id session_date session_time  load_kg  lifts_total  \
0   set_0001          P0001   2025-09-16  17:14+05:30     20.0            5   
1   set_0002          P0001   2025-09-16  17:25+05:30     20.0            5   
2   set_0003          P0002   2025-09-17  18:50+05:30     20.0            5   
3   set_0004          P0003   2025-10-01  18:40+05:30     20.0            5   
4   set_0005          P0003   2025-10-01  18:43+05:30     20.0            5   
5   set_0006          P0004   2025-10-01  19:15+05:30     20.0            5   
6   set_0007          P0004   2025-10-01  19:18+05:30     20.0            5   
7   set_0008          P0005   2025-10-30  12:56+05:30     20.0            5   
8   set_0009          P0005   2025-10-30  13:04+05:30     20.0            5   
9   set_0010          P0006   2025-10-30  13:39+05:30     20.0            5   
10  set_0011          P0006   2025-10-30  13:43+05:30     20.0            5   
11  set_0012          P0007   2026-05-08  14:08+05:3

In [8]:
# @title 📡 26-IMU Device Summary (`device_summary.csv`)
display_table(tables["devices"], clean_columns=False)

   device_id  imu_index short_code          anatomical_name    side     joint  \
0         01          1        RKP      right_knee_proximal   right      knee   
1         01          2        RHD         right_hip_distal   right       hip   
2         02          1        REP     right_elbow_proximal   right     elbow   
3         02          2        RSD    right_shoulder_distal   right  shoulder   
4         03          1        LKP       left_knee_proximal    left      knee   
5         03          2        LHD          left_hip_distal    left       hip   
6         04          1        LEP      left_elbow_proximal    left     elbow   
7         04          2        LSD     left_shoulder_distal    left  shoulder   
8      05/11          1        LHP        left_hip_proximal    left       hip   
9      05/11          2        RHP       right_hip_proximal   right       hip   
10     05/11          3         LS              lower_spine  center     spine   
11        06          1     

In [9]:
# @title ⏱️ Phase Markers Summary (`phase_markers_summary.csv`)
display_table(tables["phase_markers"], clean_columns=False)

        set_id  lift_id    marker_name  n_annotators  mean_frame  sd_frames  \
0     set_0001        1          start             3      0.0000     0.0000   
1     set_0001        1          setup             3     36.3333     4.0415   
2     set_0001        1     first_pull             3    137.3333     0.5774   
3     set_0001        1    second_pull             3    151.0000     3.0000   
4     set_0001        1          clean             3    157.3333     2.5166   
...        ...      ...            ...           ...         ...        ...   
2947  set_0054        3  jerk_recovery             3   1355.6667     4.9329   
2948  set_0054        3  overhead_hold             3   1409.3333     0.5774   
2949  set_0054        3        descend             3   1432.3333     1.5275   
2950  set_0054        3      post_rest             3   1492.6667     5.0332   
2951  set_0054        3            end             3   1515.0000     0.0000   

       sd_ms  frame  imu_idx     time_s  
0       0

In [10]:
# @title 📖 Data Dictionary (`data_dictionary.csv`)
display_table(tables["data_dictionary"], clean_columns=False)

                         file             field    dtype         unit  \
0     participant_summary.csv    participant_id      str            —   
1     participant_summary.csv         age_years      int        years   
2     participant_summary.csv               sex      str            —   
3     participant_summary.csv          height_m    float            m   
4     participant_summary.csv      body_mass_kg    float           kg   
5     participant_summary.csv         bmi_kg_m2    float       kg/m^2   
6     participant_summary.csv  experience_years    float        years   
7     participant_summary.csv       achievement      str            —   
8     participant_summary.csv    best_weight_kg    float           kg   
9     participant_summary.csv             notes      str            —   
10           sets_summary.csv            set_id      str            —   
11           sets_summary.csv    participant_id      str            —   
12           sets_summary.csv      session_date    

### 📦 Inspect Per-Set `.h5` File (Metadata, Participant Metadata & DataFrames)

In [11]:
# @title 📦 Load Set HDF5 & Inspect Structure { display-mode: "form" }
import h5py
import pandas as pd

set_id = "set_0001" # @param {type:"string"}
sd = load_set_h5(dataset_dir, set_id)

with h5py.File(sd["h5_path"], "r") as f:
    print("HDF5 top-level groups:", list(f.keys()))

print(f"accel shape: {sd['accel'].shape} | gyro shape: {sd['gyro'].shape} | time_s: {sd['time_s'].shape}")
print(f"sensors ({len(sd['sensors'])}): {sd['sensors']}")

HDF5 top-level groups: ['imu_data', 'metadata', 'participant_metadata', 'phase_markers']
accel shape: (6500, 26, 3) | gyro shape: (6500, 26, 3) | time_s: (6500,)
sensors (26): ['RKP', 'RHD', 'REP', 'RSD', 'LKP', 'LHD', 'LEP', 'LSD', 'LHP', 'RHP', 'LS', 'RWD', 'RWP', 'RED', 'LWD', 'LWP', 'LED', 'LAD', 'LAP', 'LKD', 'LSP', 'RSP', 'US', 'RAD', 'RAP', 'RKD']


In [12]:
# @title 📋 Set `/metadata` & `/participant_metadata` Attributes
print("--- /metadata ---")
display_table(pd.DataFrame(list(sd["metadata"].items()), columns=["attribute", "value"]), clean_columns=False)

print("--- /participant_metadata ---")
display_table(pd.DataFrame(list(sd["participant_metadata"].items()), columns=["attribute", "value"]), clean_columns=False)

--- /metadata ---


                 attribute                                        value
0               duration_s                                         65.0
1               has_faults                                        False
2               imu_faulty  RAD (missing); RAP (missing); RKD (missing)
3                imu_fs_hz                                        100.0
4              imu_samples                                         6500
5              lifts_total                                            5
6             lifts_usable                                            5
7                  load_kg                                         20.0
8   original_experiment_id                             experiments_0016
9           participant_id                                        P0001
10              sensors_ok                                           23
11           sensors_total                                           26
12            session_date                                   202

--- /participant_metadata ---


           attribute                                   value
0        achievement                                        
1          age_years                                      24
2     best_weight_kg                                    90.0
3          bmi_kg_m2                                   25.69
4       body_mass_kg                                    76.0
5   experience_years                                     0.0
6           height_m                                    1.72
7               name                            Enuka Mendis
8              notes  Participation in Other sports: Kabaddi
9     participant_id                                   P0001
10               sex                                       M

In [13]:
# @title 📈 IMU DataFrame (`sd["imu_df"]` — first 10 rows)
display_table(sd["imu_df"].head(10), clean_columns=False)

   sample_idx  time_s  RKP_AX  RKP_AY  RKP_AZ  RKP_GX  RKP_GY  RKP_GZ  RHD_AX  \
0           0    0.00  -9.725   1.938   1.653   5.457  -0.030   5.457  -9.323   
1           1    0.01  -9.643   1.856   1.609   4.512  -0.976   6.402  -9.336   
2           2    0.02  -9.641   1.713   1.636   3.963  -1.646   7.043  -9.461   
3           3    0.03  -9.735   1.715   1.653   4.360  -1.402   7.652  -9.534   
4           4    0.04  -9.787   1.757   1.641   4.177  -1.037   8.567  -9.494   
5           5    0.05  -9.752   1.785   1.610   3.598  -0.823   9.512  -9.383   
6           6    0.06  -9.687   1.735   1.598   3.201  -0.884   9.878  -9.248   
7           7    0.07  -9.681   1.662   1.642   3.598  -1.128   9.787  -9.266   
8           8    0.08  -9.568   1.413   1.628   3.384  -0.549   9.055  -9.025   
9           9    0.09  -9.488   1.397   1.665   2.043  -0.427   7.165  -9.127   

   RHD_AY  ...  RAP_AZ  RAP_GX  RAP_GY  RAP_GZ  RKD_AX  RKD_AY  RKD_AZ  \
0   3.027  ...     NaN     NaN    

In [14]:
# @title ⏱️ Phase Markers — Consensus (`sd["consensus_df"]`) & Annotators (`sd["annotators_df"]`)
print("--- consensus_df ---")
display_table(sd["consensus_df"], clean_columns=False)

print("--- annotators_df (first 20 rows) ---")
display_table(sd["annotators_df"].head(20), clean_columns=False)

--- consensus_df ---


    frame  imu_idx  lift_id        marker_name  mean_frame  n_annotators  \
0       0        0        1              start      0.0000             3   
1      36      120        1              setup     36.3333             3   
2     137      457        1         first_pull    137.3333             3   
3     151      503        1        second_pull    151.0000             3   
4     157      523        1              clean    157.3333             3   
5     230      767        1  intermediate_rest    230.3333             3   
6     257      857        1         jerk_split    257.0000             3   
7     338     1127        1      jerk_recovery    337.6667             3   
8     361     1203        1      overhead_hold    361.3333             3   
9     362     1207        1            descend    362.0000             3   
10    448     1493        1          post_rest    448.3333             3   
11    495     1650        1                end    494.6667             3   
12    448   

--- annotators_df (first 20 rows) ---


   annotator_id  frame  imu_idx  lift_id        marker_name     time_s
0            a2      0        0        1              start   0.000000
1            a2     37      123        1              setup   1.233333
2            a2    138      460        1         first_pull   4.600000
3            a2    151      503        1        second_pull   5.033333
4            a2    157      523        1              clean   5.233333
5            a2    225      750        1  intermediate_rest   7.500000
6            a2    257      857        1         jerk_split   8.566667
7            a2    336     1120        1      jerk_recovery  11.200000
8            a2    359     1197        1      overhead_hold  11.966667
9            a2    360     1200        1            descend  12.000000
10           a2    442     1473        1          post_rest  14.733333
11           a2    495     1650        1                end  16.500000
12           a2    442     1473        2              start  14.733333
13    